# Tag 8

- DCL Demo
  - Voraussetzung: `CREATE LOGIN` (Serverebene), `CREATE USER` (Datenbankebene), `CREATE SCHEMA`
  - `GRANT` (Recht gewähren)
  - `DENY` (Recht verweigern)
  - `REVOKE` (Recht entziehen)
- Row Level Security
  - Zeilenweise Überprüfung, wer die Daten sehen darf.
  - Erfordert:
    - In jeder Datenzeile einen Wert mit dem gefiltert werden kann, z.B. Username
    - Filterfunktion, die 1 zurückgibt, wenn die Zeile angezeigt werden soll
    - Sicherheitsrichtlinie, die die Filterfunktion an die Tabelle bindet (SECURITY POLICY)
- Dynamic Data Masking
  - Maskierung sensibler Daten

## Nachmittag

- Nachtrag: `STRING_AGG`
- Fragestunde zu Töpfermarkt Übungen
- Neue Übungsreihe BikeShop

## STRING_AGG
 
- Zeichenkettenfunktion, die mehrere Werte durch ein Trennzeichen getrennt ausgibt.
- Diese Funktion hebelt quasi die 1. Normalform aus, indem Sie mehrere Werte in eine einzelne Zelle der Ausgabe schreibt.
- Referenz: [STRING_AGG (Transact-SQL) - SQL Server](STRING_AGG (Transact-SQL) - SQL Server)
 
```sql
STRING_AGG ( expression , separator ) [ <order_clause> ]
 
<order_clause> ::=
    WITHIN GROUP ( ORDER BY <order_by_expression_list> [ ASC | DESC ] )

`STRING_AGG` in NormalisierungsDemo

Zeige Personal Vorname + Nachname, Projekte in einer Zelle

`STRING_AGG` in Töpfermarkt

Ortsname, Marktbezeichnung, Termine in einer Zelle absteigend (201 Zeilen)

Produktart untereinander (Becher, Krug, ...), Erhältliche Farben in einer Zelle alphabetisch

Wie verhinder ich, dass Farben doppelt angezeigt werden?

In [ ]:
USE NormalisierungsDemo
SELECT
  CONCAT(p.Vorname, ' ', p.Name) AS [Personal],
  STRING_AGG(pj.Beschreibung, ', ') AS Projekte
FROM Personal AS p
LEFT JOIN Arbeitet_An AS aa ON p.PersonalNr = aa.PersonalNr
LEFT JOIN Projekt AS pj ON aa.ProjNr = pj.ProjNr
GROUP BY p.Vorname, p.Name

(5 rows affected)

Personal       | Projekte                                       
---------------+------------------------------------------------
Sean Conolly   | Kundenumfrage, Verkaufsmesse, Konkurrenzanalyse
Maria Eichenau | Verkaufsmesse                                  
Lutz Frawley   | Verkaufsmesse                                  
Stefanie Glahn | Kundenumfrage, Konkurrenzanalyse               
Karin Kirsch   | Kundenumfrage, Konkurrenzanalyse               
(5 rows)

Total execution time: 00:00:00.063

In [52]:
USE Toepfermarkt;
SELECT
    m.Ort,
    m.Marktbezeichnung,
    STRING_AGG(CONVERT(varchar, t.Von, 104), ', ')
        WITHIN GROUP (ORDER BY t.Von DESC) AS Termine
FROM Markt AS m
JOIN Termin AS t ON m.MarktId = t.MarktId
GROUP BY m.MarktId, m.Ort, m.Marktbezeichnung
ORDER BY m.Ort;

(201 rows affected)

Ort                                | Marktbezeichnung                        | Termine                                       
-----------------------------------+-----------------------------------------+-----------------------------------------------
Adlersberg/Regensburg              | Töpfermarkt Prösslbräu Adlersberg       | 17.08.2019                                    
Ahorn                              | Töpfermarkt                             | 08.06.2019                                    
Alzey                              | Töpfermarkt Alzey                       | 09.11.2018                                    
Amberg                             | Töpfermarkt                             | 28.04.2018                                    
Aschaffenburg                      | Töpfermarkt Aschaffenburg, Johannisburg | 27.04.2019                                    
Au am Inn, Lkr. Mühldorf           | Töpfermarkt Kloster Au                  | 26.10.2019        

In [43]:
SELECT compatibility_level
FROM sys.databases
WHERE name = 'Toepfermarkt';

(1 row affected)

compatibility_level
-------------------
160                
(1 row)

Total execution time: 00:00:00.056

In [42]:
ALTER DATABASE Toepfermarkt SET COMPATIBILITY_LEVEL = 160;

Commands completed successfully.

Total execution time: 00:00:00.022

In [87]:
SELECT
    Produktart,
    STRING_AGG(Farbe, ', ') WITHIN GROUP (ORDER BY Farbe) AS Farben
FROM (
    SELECT DISTINCT
        pa.ProduktartId,
        pa.Bezeichnung AS Produktart,
        f.Bezeichnung AS Farbe
    FROM Produkt AS p
    JOIN Produktart AS pa ON p.Produktart = pa.ProduktartId
    JOIN Farbe AS f ON p.Farbe = f.FarbeId
) AS DistinktFarben
GROUP BY ProduktartId, Produktart;

(7 rows affected)

Produktart | Farben                                       
-----------+----------------------------------------------
Becher     | Gelb, Grün, Rot, Schwarz                     
Krug       | Blau, Gelb, Grün, Rot, Schwarz               
Schale     | Blau, Gelb, Grün, Rot, Schwarz               
Tasse      | Blau, Gelb, Grün, Rot, Schwarz, Ultra Violett
Teller     | Blau, Gelb, Grün, Rot, Schwarz               
Topf       | Blau, Gelb, Grün, Rot, Schwarz               
Vase       | Blau, Gelb, Grün, Rot, Schwarz               
(7 rows)

Total execution time: 00:00:00.180

In [ ]:
USE Toepfermarkt;

WITH DistinktFarben AS (
    SELECT DISTINCT
        pa.ProduktartId,
        pa.Bezeichnung AS Produktart,
        f.Bezeichnung AS Farbe
    FROM Produkt AS p
    JOIN Produktart AS pa ON p.Produktart = pa.ProduktartId
    JOIN Farbe AS f ON p.Farbe = f.FarbeId
)
SELECT
    Produktart,
    STRING_AGG(Farbe, ', ') WITHIN GROUP (ORDER BY Farbe) AS Farben
FROM DistinktFarben
GROUP BY ProduktartId, Produktart;

(7 rows affected)

Produktart | Farben                                       
-----------+----------------------------------------------
Becher     | Gelb, Grün, Rot, Schwarz                     
Krug       | Blau, Gelb, Grün, Rot, Schwarz               
Schale     | Blau, Gelb, Grün, Rot, Schwarz               
Tasse      | Blau, Gelb, Grün, Rot, Schwarz, Ultra Violett
Teller     | Blau, Gelb, Grün, Rot, Schwarz               
Topf       | Blau, Gelb, Grün, Rot, Schwarz               
Vase       | Blau, Gelb, Grün, Rot, Schwarz               
(7 rows)

Total execution time: 00:00:00.053